## GPT prompting: n50

### requires python >= 3.10


In [1]:
# for auto-reloading extenrnal modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [3]:
sys.path.append("../../../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [4]:
from prompts.semantic_categories.v02.prompt import SYSTEM_PROMPT, FEW_SHOTS_STR, FEW_SHOTS

from prompts.semantic_categories.v03.prompt import (
    ABSTRACT_SYSTEM_PROMPT, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_FEW_SHOTS, 
)

from prompts.semantic_categories.v04.prompt import (
    ALIVE_SYSTEM_PROMPT, ALIVE_FEW_SHOTS_STR, ALIVE_FEW_SHOTS, 
    EVENT_SYSTEM_PROMPT, EVENT_FEW_SHOTS_STR, EVENT_FEW_SHOTS, 
    TIME_SYSTEM_PROMPT, TIME_FEW_SHOTS_STR, TIME_FEW_SHOTS,
    ORG_SYSTEM_PROMPT, ORG_FEW_SHOTS_STR, ORG_FEW_SHOTS
)

In [5]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [6]:
RESULTS_DIR = "../../results/"

# kuhu salvestub loc prompti tulemus
LOC_SUB_DIR = RESULTS_DIR + "n50_examples_large_v01/gpt_v01/"
# kuhu salvestada ülejäänud promptide tulemused
OTHER_SUB_DIR = RESULTS_DIR + "n50_examples_large_v01/gpt_v02/"

# andmefail
DATA_FILE = "../../data/n50_examples_large_v01.csv"

# loc prompti tulemusfail
GPT_LOC_ANSWER_FILE = LOC_SUB_DIR + "gpt_b10_run01.csv"
# ülejäänud promptide tulemusfailid
GPT_ANSWER_FILE_ALIVE = OTHER_SUB_DIR + "gpt_b10_run01_no_is_alive.csv"
GPT_ANSWER_FILE_EVENT = OTHER_SUB_DIR + "gpt_b10_run01_no_is_event.csv"
GPT_ANSWER_FILE_TIME = OTHER_SUB_DIR + "gpt_b10_run01_no_is_timex.csv"
GPT_ANSWER_FILE_ORG = OTHER_SUB_DIR + "gpt_b10_run01_no_is_org.csv"
GPT_ANSWER_FILE_ABSTRACT = OTHER_SUB_DIR + "gpt_b10_run01_no_is_abstract.csv"

# fail ülejäänud promptide tulemustega kokku (alive+event+time+org+abstract)
GPT_FILTERED_FILE = OTHER_SUB_DIR + "gpt_b10_run01_no_filtered.csv"

# fail kõigi vastustega (kõik laused, tekitatud uus classification veerg)
GPT_ANSWER_FILE = OTHER_SUB_DIR + "gpt_b10_run01.csv"

# väike sample fail 100 näitega
GPT_ANSWER_FILE_SAMP = OTHER_SUB_DIR + "gpt_b10_run01_sample.csv"

CONF_FILE = "../../../../v04_verb-case_pattern/minu_code/azure.ini"


# OSA I : Andmed

In [7]:
# algne andmefail
df1 = pd.read_csv(DATA_FILE, encoding="utf-8",  sep=",")

spatial_obl_ex = df1.iloc[:10000]
spatial_obl_ex = spatial_obl_ex.sample(frac=1)

In [8]:
spatial_obl_ex

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag
3235,14682666,23144397,12,tooma,kaasa,all,skeidi,skeidile,Noppige siis spordipoodidest flaiereid ja täitke need ära ning tooge õhtul skeidile kaasa !,NaN,NaN,NaN
9666,4675269,7512032,10,heitlema,NaN,in,Nagano,Naganos,"1998 arvati ta olümplaseks , kuid kui teised heitlesid Naganos , siis tema pidas oma mängud kodus .",NaN,location,LOC
18,21087297,29915194,3,lootma,NaN,in,Marko,Markos,"moRt: Markos looda saa , kasutab minu arust majapidamisriistana ...",NaN,NaN,LOC
2945,11624583,18609145,6,eelistama,NaN,all,jäätis,jäätistele,"Kuigi 10-aastane Laura eelistab eesti jäätistele väljamaiseid , ei ütle ta kunagi ära ka kodumaisest jäätisest .",NaN,NaN,NaN
2357,966364,1539316,9,vähendama,NaN,all,sina,teile,"Mis saab fondist edasi , kui Soros vähendas teile antavat toetust 2/3 võrra ?",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
8769,981021,1562806,8,levima,NaN,ad,selgitus,selgitusel,"Hansapanga suhtekorraldajate Kristi Liiva ja Ando Noormetsa selgitusel levis tehnilise rikke versioon esmaspäeva hommikul seetõttu , et panga klienditoe töötajatel ei olnud enne turvajuurdluse lõppu juhtunu kohta täpset infot .",NaN,NaN,NaN
5307,15749627,24570642,10,saatma,NaN,el,laevastik,laevastikust,"Ameerika saadab Tallinna linnapea kutse peale muidugi jupikese oma laevastikust Tallinna reidile , aga emalaev jääb Sundi kinni .",NaN,location,NaN
770,974771,1552554,7,tõttama,NaN,all,kolleeg,kolleegidele,Samuti olevat kõik Austraalia lõbutüdrukud Sydney kolleegidele appi tõtanud .,NaN,alive,NaN
1231,4825664,7748191,4,hüppama,NaN,el,loodetud,loodetust,"Mitmedki mehed hüppasid loodetust kümmekond meetrit vähem , kuni Masahiko Harada tõusis 115meetrise hüppega teiseks .",NaN,NaN,NaN


# OSA II : GPT

## GPT jaoks vajalik

In [9]:
config = configparser.ConfigParser()

status = config.read(CONF_FILE) 
assert status == [CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [10]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

## Functions for classifying and explanation

In [11]:

def classify_batch(my_batch, few_shots, system_prompt, client, deployment):
    """Gets a yes/no answer for a batch of sentences and phrases. 
    """
    #print("classify", len(my_batch))
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": few_shots,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":  json.dumps(user_payload, ensure_ascii=False)}
        ]

        #return None, None
        response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(my_batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(my_batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


In [12]:
# kui enamus vastuseid peaks olema "yes" ehk location, siis küsida "no" puhul põhjendust
# kasutada n80 puhul
def explain_non_locations(
    client, 
    deployment,
    batch: List[Dict[str, str]],
    yes_no_results: List[str],
    subset_ratio: float = 0.0,

) -> Dict[int, str]:

    # Determine which indices to explain
    no_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "no"]
    yes_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "yes"]

    # Diagnostic subset
    diag_count = int(len(yes_indices) * subset_ratio)
    diag_indices = yes_indices[:diag_count]

    explain_indices = no_indices + diag_indices
    if not explain_indices:
        return None, None, None

    items_to_explain = [
        {
            "index": i,
            "l": json.loads(batch[i])["l"],
            "c": json.loads(batch[i])["c"],
            "classification": yes_no_results[i]
        }
        for i in explain_indices
    ]

    messages = [
        {"role": "system", 
         "content": ("Explain why each phrase 'c' was classified as adverbial of place ('yes') or not adverbial of place ('no') in sentence 'l'." 
                       "Give one sentence answer."
                        "You MUST return only a pure JSON object, without markdown and code fences. "
                        "The output must be a mapping: {index: explanation}. "
                        "Do not include ```json or any backticks. Do not include commentary.")
        },
        {"role": "user", "content": (
            """For EACH item without missing any, return a JSON object mapping index → explanation in this format '{"0": "explanation", "3": "explanation"}'.\n"""
            "Items:\n" +  json.dumps(items_to_explain, ensure_ascii=False)
        )}
    ]
    #return None, None,None 
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
    )

    raw = response.choices[0].message.content.strip()

    # ---- Pydantic validation ----
    try:
        ClassificationAnswer(form = json.loads(raw))
    except ValidationError as e:
        raise ValueError(f"Invalid JSON structure returned in explanations:\n{e}")

    return response, raw, explain_indices



# kui enamus vastuseid peaks olema "no" ehk mitte location siis küsime "yes" puhul põhjendust
# kasutada n20 puhul
def explain_locations(
    client, 
    deployment,
    batch: List[Dict[str, str]],
    yes_no_results: List[str],
    subset_ratio: float = 0.0,

) -> Dict[int, str]:

    # Determine which indices to explain
    no_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "no"]
    yes_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "yes"]

    # Diagnostic subset (sanity check for "no" answers)
    diag_count = int(len(no_indices) * subset_ratio)
    diag_indices = no_indices[:diag_count]

    explain_indices = yes_indices + diag_indices
    if not explain_indices:
        return None, None, None

    items_to_explain = [
        {
            "index": i,
            "l": json.loads(batch[i])["l"],
            "c": json.loads(batch[i])["c"],
            "classification": yes_no_results[i]
        }
        for i in explain_indices
    ]

    messages = [
        {"role": "system", 
         "content": ("Explain why each phrase 'c' was classified as adverbial of place ('yes') or not adverbial of place ('no') in sentence 'l'." 
                       "Give one sentence answer."
                        "You MUST return only a pure JSON object, without markdown and code fences. "
                        "The output must be a mapping: {index: explanation}. "
                        "Do not include ```json or any backticks. Do not include commentary.")
        },
        {"role": "user", "content": (
            """For EACH item without missing any, return a JSON object mapping index → explanation in this format '{"0": "explanation", "3": "explanation"}'.\n"""
            "Items:\n" +  json.dumps(items_to_explain, ensure_ascii=False)
        )}
    ]
    #return None, None,None 
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
    )

    raw = response.choices[0].message.content.strip()

    # ---- Pydantic validation ----
    try:
        ClassificationAnswer(form = json.loads(raw))
    except ValidationError as e:
        raise ValueError(f"Invalid JSON structure returned in explanations:\n{e}")

    return response, raw, explain_indices

# LOCATION

## NB! muuda max_allowed_tok ja explain_* funktsiooni kui vaja

In [13]:
df = spatial_obl_ex.copy()

In [14]:
len(df)

10000

In [15]:
results = []
results2 = []
responses = []
explanations = []
explanations_all = {}

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10

# kui suure osa võtta "yes" vastustest "why" küsimusse
# kui on 0.2, siis võiks max 2/10 olla põhjendatud
subset_ratio = 0.2
batch_start_index = 0

# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 4800000


batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, FEW_SHOTS_STR, SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    # võtab välja kõik batchis olnud "no" ja "yes" ja küsib why
    # NB! muuta funktsiooni kas explain_non_locations või explain_locations
    expl_response, batch_explanations, answered_idx = explain_non_locations(
            batch=batch,
            yes_no_results=result_yesno,
            subset_ratio=subset_ratio,
            client=client, 
            deployment=DEPLOYMENT
        )

    # mapping: vastused õige lause+fraasiga kokku
    if batch_explanations is not None:
        used_tokens += expl_response.usage.total_tokens
        
        explanations.append(json.loads(batch_explanations))
        
        # Map batch-local -> global indices
        for local_i, explanation in json.loads(batch_explanations).items():
            global_i = batch_start_index + int(local_i)
            explanations_all[global_i] = explanation

    batch_start_index += len(batch)

    #if used_tokens >= max_allowed_tok:
    #    print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
    #    break    

1000it [38:15,  2.30s/it]


In [19]:
used_tokens

3929963

In [16]:
len(results)

10000

In [17]:
# andmed tabelisse

faulty_batches = {}
faulty_answers = {}
num_full_batches = int(len(df)/bs)
partial_batches = False if num_full_batches*bs == len(df) else True

if len(results) == len(df):
    df["classification"] = [r["a"] for r in results]

    new_explanations = []
    for i in range(len(df)):
        if i in explanations_all.keys():
            new_explanations.append( explanations_all[i])
        else:
            new_explanations.append("")
    
        
    df["explanation"] = new_explanations 


else: # juhuks kui mudel ei anna õiget arvu vastuseid tagasi
    new_results = []
    new_explanations = []
    for b, (batchres, expl) in enumerate(zip(results2, explanations),start=0):
        expected_len = bs if b < num_full_batches else len(df)-(num_full_batches*bs)
        
        if len(batchres) != expected_len and b < num_full_batches: # pole poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(expected_len)]
            #print(num_full_batches, b, len(replacement))
            new_results += replacement
            new_explanations += replacement
            faulty_batches[b] = batchres
            faulty_answers[b] = expl
        elif len(batchres) != expected_len and b >= num_full_batches and partial_batches:  # on poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(len(expected_len))]
            new_results += replacement
            faulty_batches[b] = batchres
            new_explanations += replacement
            faulty_answers[b] = expl
        elif len(batchres) == expected_len: # kõik ok 
            new_results += [r["a"] for r in batchres]
            for i in range(len(batchres)):
                if str(i) in expl.keys():
                    new_explanations.append( expl[str(i)])
                else:
                    new_explanations.append("")
            
    
    df["classification"] = new_results
    df["explanation"] = new_explanations


In [18]:
df.to_csv(GPT_LOC_ANSWER_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [22]:
print("yes:", len(df[df["classification"]=="yes"]))
print("no:", len(df[df["classification"]=="no"]))

yes: 6336
no: 3664


# ABSTRACT LOCATION

In [24]:
df_0 = df[df["classification"]=="no"].copy()

## NB! muuda max_allowed_tok kui vaja

In [26]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 2500000

rows = df_0.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

367it [04:12,  1.45it/s]


In [27]:
used_tokens

600701

In [28]:
len(results)

3664

In [29]:
if len(results) == len(df_0):
    df_0["is_abstract"] = [r["a"] for r in results]

In [30]:
df_0.to_csv(GPT_ANSWER_FILE_ABSTRACT, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [32]:
print(len(df_0[df_0["is_abstract"]=="yes"]))
print(len(df_0[df_0["is_abstract"]=="no"]))

578
3086


# ALIVE

## NB! muuda max_allowed_tok kui vaja

In [77]:
#df_1 = df_0[df_0["is_abstract"]=="no"].copy()
df_1 = df[df["classification"]=="no"].copy()

In [79]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0

# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 1500000

rows = df_1.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ALIVE_FEW_SHOTS_STR, ALIVE_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

367it [04:37,  1.32it/s]


In [80]:
used_tokens # 345468

412063

In [81]:
len(results)

3664

In [82]:
if len(results) == len(df_1):
    df_1["is_alive"] = [r["a"] for r in results]

In [83]:
df_1.to_csv(GPT_ANSWER_FILE_ALIVE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [84]:
print(len(df_1[df_1["is_alive"]=="yes"]))
print(len(df_1[df_1["is_alive"]=="no"]))

892
2772


# TIMEX

In [85]:
#df_3 = df_1[df_1["is_alive"]=="no"].copy()
df_3 = df[df["classification"]=="no"].copy()

## NB! muuda max_allowed_tok kui vaja

In [87]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 1500000

rows = df_3.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, TIME_FEW_SHOTS_STR, TIME_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

367it [04:07,  1.49it/s]


In [88]:
used_tokens 

396263

In [89]:
len(results)

3664

In [90]:
if len(results) == len(df_3):
    df_3["is_time"] = [r["a"] for r in results]

In [91]:
df_3.to_csv(GPT_ANSWER_FILE_TIME, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [92]:
print(len(df_3[df_3["is_time"]=="yes"]))
print(len(df_3[df_3["is_time"]=="no"]))

356
3308


# EVENT

In [50]:
df_2 = df_3[df_3["is_time"]=="no"].copy()
#df_2 = df[df["classification"]=="no"].copy()

## NB! muuda max_allowed_tok kui vaja

In [52]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 1500000

rows = df_2.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, EVENT_FEW_SHOTS_STR, EVENT_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

191it [02:25,  1.32it/s]


In [53]:
used_tokens 

286686

In [55]:
len(results)

1904

In [56]:
if len(results) == len(df_2):
    df_2["is_event"] = [r["a"] for r in results]

In [57]:
df_2.to_csv(GPT_ANSWER_FILE_EVENT, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [58]:
print(len(df_2[df_2["is_event"]=="yes"]))
print(len(df_2[df_2["is_event"]=="no"]))

227
1677


# ORG

In [95]:
#df_4 = df_2[df_2["is_event"]=="no"].copy()
df_4 = df[df["classification"]=="no"].copy()

In [96]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 900000

rows = df_4.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ORG_FEW_SHOTS_STR, ORG_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

367it [04:24,  1.39it/s]


In [97]:
used_tokens

464932

In [98]:
len(results)

3664

In [99]:
if len(results) == len(df_4):
    df_4["is_org"] = [r["a"] for r in results]

In [100]:
df_4.to_csv(GPT_ANSWER_FILE_ORG, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [101]:
print(len(df_4[df_4["is_org"]=="yes"]))
print(len(df_4[df_4["is_org"]=="no"]))

105
3559


### Kokku alive+event+time filtreeritud fail

In [102]:
fname1 = GPT_ANSWER_FILE_ABSTRACT 
fname2 = GPT_ANSWER_FILE_EVENT
fname3 = GPT_ANSWER_FILE_TIME
fname4 = GPT_ANSWER_FILE_ORG
fname5 = GPT_ANSWER_FILE_ALIVE

df1 = pd.read_csv(fname1, encoding="utf-8",  sep=",")
df2 = pd.read_csv(fname2, encoding="utf-8",  sep=",")
df3 = pd.read_csv(fname3, encoding="utf-8",  sep=",")
df4 = pd.read_csv(fname4, encoding="utf-8",  sep=",")
df5 = pd.read_csv(fname5, encoding="utf-8",  sep=",")

key_cols = ['sentence_id','head_id', "verb", "verb_compound", "morph_case", "form"]

df2_selected = df2[key_cols + ["is_event"]].copy()
df3_selected = df3[key_cols + ["is_time"]].copy()
df4_selected = df4[key_cols + ["is_org"]].copy()
df5_selected = df5[key_cols + ["is_alive"]].copy()

df1['verb_compound'] = df1['verb_compound'].astype('string').str.strip()
df2_selected['verb_compound'] = df2_selected['verb_compound'].astype('string').str.strip()
df3_selected['verb_compound'] = df3_selected['verb_compound'].astype('string').str.strip()
df4_selected['verb_compound'] = df4_selected['verb_compound'].astype('string').str.strip()
df5_selected['verb_compound'] = df5_selected['verb_compound'].astype('string').str.strip()

# Merge df1 with df2_selected etc
merged_df = df1.merge(df2_selected, on=key_cols, how='left')

merged_df2 = merged_df.merge(df3_selected, on=key_cols, how='left')

merged_df3 = merged_df2.merge(df4_selected, on=key_cols, how='left')

final_df = merged_df3.merge(df5_selected, on=key_cols, how='left')

cols = ['is_time', 'is_event', 'is_alive', 'is_org', 'is_abstract']

final_df[cols] = final_df[cols].fillna('no')
final_df['verb_compound'] = final_df['verb_compound'].fillna('')

In [103]:
final_df.to_csv(GPT_FILTERED_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

## Kokku kõikide varasemate klassifikatsioonidega + classification3 loomine

### seda osa saab korrata ilma gpt osa uuesti tegemata

In [104]:
df1 = pd.read_csv(GPT_LOC_ANSWER_FILE, encoding="utf-8", sep=",")
df2 = pd.read_csv(GPT_FILTERED_FILE, encoding="utf-8", sep=",")

In [105]:
cols = ["head_id", "form", "verb", "verb_compound", "morph_case","sentence_id", "is_time", "is_alive", "is_event", "is_org", "is_abstract"]
df3 = df2[cols]

In [106]:
filter3 = pd.merge(df1, df3, on=["head_id", "form", "verb", "verb_compound", "morph_case", "sentence_id"], how='left')


""" kas saab klassifitseerimisel korraks kõrvale jätta
 see on yes/no selleks, et lihtsamalt näha, kas peale eeldefineeritud klasside saab veel midagi välja võtta
"""

def exclude(row):
    
    # kui eelmine tulemus = "yes" -> jääb "yes"  -> saame välja visata
    if row["classification"] == "yes":
        return "yes"
    
    # kui on aeg -> "yes" -> saame välja visata
    if row["is_time"] == "yes":
        return "yes"
    
    # event -> "yes" -> saame välja visata
    if row["is_event"] == "yes":
        return "yes"
    
    # elus -> "yes" -> saame välja visata
    if row["is_alive"] == "yes":
        return "yes"
    
    # org -> "yes" -> saame välja visata
    if row["is_org"] == "yes":
        return "yes"

    
    # kui oli "no", NaN ja/või alive/time/abstract kõik olid "no"
    else:
        return "no"

In [90]:
#filter3["exclude"] = filter3.apply(exclude, axis=1)

In [107]:
# uus classification2
# muuta vastavalt vajadusele

def new_class(row):
    
    # kui eelmine tulemus = "yes" -> jääb "yes"  -> saame välja visata
    if row["classification"] == "yes":
        return "loc"

    if row["is_abstract"] == "yes":
        return "loc"
    
    # kui on aeg -> "yes" -> saame välja visata
    if row["is_time"] == "yes":
        return "time"
    
    # event -> "yes" -> saame välja visata
    if row["is_event"] == "yes":
        return "event"
    
    # elus -> "yes" -> saame välja visata
    if row["is_alive"] == "yes":
        return "actor"
    
    # org -> "yes" -> saame välja visata
    if row["is_org"] == "yes":
        return "actor"

    
    # kui oli "no", NaN ja/või alive/time/abstract kõik olid "no"
    else:
        return "UNK"

In [108]:
filter3["classification2"] = filter3.apply(new_class, axis=1)

In [109]:
filter3.to_csv(GPT_ANSWER_FILE, encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [110]:
filter3_2 = filter3.iloc[:100]

In [111]:
filter3_2.to_csv(GPT_ANSWER_FILE_SAMP, encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)